[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/35_reinforcement_learning/04_dqn_and_policy_gradient.ipynb)

# Bài 04 — Deep Q-Network và Policy Gradient

**🎯 Mục tiêu**
- **Xấp xỉ hàm**: thay bảng $Q$ bằng mạng nơ-ron cho trạng thái liên tục (CartPole).
- **DQN**: experience replay, target network, Double DQN — cài from scratch; thấy rõ sự **bất ổn** của deep RL qua nhiều seed.
- **Policy gradient** (REINFORCE): tối ưu trực tiếp chính sách; vai trò của **baseline** trong giảm phương sai.
- Bản đồ các thuật toán hiện đại: actor-critic, PPO, và RLHF cho LLM.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/35_reinforcement_learning

In [ ]:
import copy
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.envs import CartPole
from ml_course.visualization import set_style

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Bài này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
torch.set_num_threads(1)                                               # mạng rất nhỏ: CPU một luồng nhanh và tái lập được
print("PyTorch trên CPU (mạng nhỏ, cập nhật từng bước — GPU không nhanh hơn)")

## 1. CartPole: trạng thái liên tục

Một xe đẩy mang con lắc ngược; mỗi bước chọn đẩy trái/phải. Trạng thái 4 số thực (vị trí, vận tốc xe, góc, vận tốc góc); +1 cho mỗi bước con lắc còn đứng (|góc| ≤ 12°, |vị trí| ≤ 2.4), tối đa 500 bước. Không thể lập bảng cho trạng thái liên tục — cần **xấp xỉ hàm**.

In [ ]:
def run_episode(env, policy, seed):
    s, done, total = env.reset(seed=seed), False, 0.0
    while not done:
        s, r, done = env.step(policy(s))
        total += r
    return total


env = CartPole()
rng = np.random.default_rng(0)
print(f"chính sách ngẫu nhiên: return trung bình = {np.mean([run_episode(env, lambda s: int(rng.integers(2)), seed=i) for i in range(100)]):.1f}")
print(f"luật tay 'đẩy về phía con lắc đổ': {np.mean([run_episode(env, lambda s: int(s[2] > 0), seed=i) for i in range(100)]):.1f}")
print(f"luật tay 'góc + vận tốc góc': {np.mean([run_episode(env, lambda s: int(s[2] + s[3] > 0), seed=i) for i in range(100)]):.1f}")


def evaluate(policy, episodes=10, seed=10_000):
    return np.mean([run_episode(CartPole(), policy, seed + i) for i in range(episodes)])

Lưu ý trung thực: một **luật tay** một dòng ("đẩy theo hướng góc + vận tốc góc") đã gần như giữ được 500 bước — CartPole là bài **dễ**. Ta dùng nó vì nhỏ và chạy nhanh để minh hoạ thuật toán; còn trong thực tế, RL đáng dùng khi **không** có luật đơn giản như vậy. Bài học chung: luôn so với baseline đơn giản (luật tay, heuristic) trước khi khẳng định RL "giải được" bài toán.

## 2. Deep Q-Network

Q-learning với $Q_\theta(s, a)$ là mạng nơ-ron (Mnih et al., 2015 — chơi Atari từ pixel). Huấn luyện ngây thơ (cập nhật trên từng bước liên tiếp, mục tiêu tính bằng chính mạng đang học) **phân kỳ** vì ba yếu tố "bộ ba chết người" (deadly triad): xấp xỉ hàm + bootstrapping + off-policy. DQN thêm hai thủ thuật:
- **Experience replay**: lưu các bước $(s, a, r, s', \text{done})$ vào bộ nhớ, huấn luyện trên **batch ngẫu nhiên** → phá tương quan giữa các mẫu liên tiếp, tái sử dụng dữ liệu.
- **Target network** $Q_{\bar\theta}$: bản sao mạng cập nhật chậm, dùng để tính mục tiêu $y = r + \gamma\max_{a'}Q_{\bar\theta}(s', a')$ → mục tiêu không "chạy theo" mạng mỗi bước.

Thêm **Double DQN** (chọn $a'$ bằng mạng chính, đánh giá bằng target — bài 03), **Huber loss**, cập nhật target **mềm** $\bar\theta \leftarrow 0.995\bar\theta + 0.005\theta$, và ε giảm tuyến tính 1 → 0.05 trong 10 000 bước. Phân biệt: kết thúc vì **thất bại** (giá trị tương lai = 0) và vì **hết giờ** (500 bước — vẫn bootstrap, vì trạng thái đó không thật sự kết thúc).

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, obs_dim):
        self.s, self.s2 = np.zeros((capacity, obs_dim), np.float32), np.zeros((capacity, obs_dim), np.float32)
        self.a, self.r, self.d = np.zeros(capacity, np.int64), np.zeros(capacity, np.float32), np.zeros(capacity, np.float32)
        self.capacity, self.n = capacity, 0

    def add(self, s, a, r, s2, terminal):
        i = self.n % self.capacity
        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, float(terminal)
        self.n += 1

    def sample(self, batch, rng):
        idx = rng.integers(0, min(self.n, self.capacity), batch)
        return tuple(torch.tensor(x[idx]) for x in (self.s, self.a, self.r, self.s2, self.d))


def train_dqn(seed, total_steps=40_000, gamma=0.99, eval_every=2500):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    env = CartPole()
    q = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
    q_target = copy.deepcopy(q)
    opt = torch.optim.Adam(q.parameters(), lr=5e-4)
    buffer = ReplayBuffer(50_000, 4)
    greedy = lambda s: int(q(torch.tensor(s, dtype=torch.float32)).argmax())
    s, curve, best_score, best_state = env.reset(seed=seed), [], -1, None
    for step in range(total_steps):
        eps = max(0.05, 1 - step / 10_000)
        if rng.random() < eps:
            a = int(rng.integers(2))
        else:
            with torch.no_grad():
                a = greedy(s)
        s2, r, done = env.step(a)
        buffer.add(s, a, r, s2, terminal=done and env.t < env.max_steps)   # hết giờ ≠ thất bại
        s = env.reset() if done else s2
        if buffer.n >= 1000:
            bs, ba, br, bs2, bd = buffer.sample(64, rng)
            with torch.no_grad():
                a_next = q(bs2).argmax(1, keepdim=True)                     # Double DQN: mạng chính chọn...
                y = br + gamma * (1 - bd) * q_target(bs2).gather(1, a_next).squeeze(1)   # ... target đánh giá
            loss = F.smooth_l1_loss(q(bs).gather(1, ba[:, None]).squeeze(1), y)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q.parameters(), 10)
            opt.step()
            with torch.no_grad():                                           # target cập nhật mềm
                for p, p_t in zip(q.parameters(), q_target.parameters()):
                    p_t.mul_(0.995).add_(0.005 * p)
        if (step + 1) % eval_every == 0:
            with torch.no_grad():
                score = evaluate(greedy)
            curve.append(score)
            if score > best_score:
                best_score, best_state = score, copy.deepcopy(q.state_dict())
    return curve, best_score, best_state


dqn_curves = {}
for seed in range(3):
    t0 = time.perf_counter()
    dqn_curves[seed], best, _ = train_dqn(seed)
    print(f"DQN seed {seed}: return đánh giá (10 lượt, tham lam) theo thời gian = {[int(v) for v in dqn_curves[seed]]}")
    print(f"             tốt nhất = {best:.0f} | cuối cùng = {dqn_curves[seed][-1]:.0f} | {time.perf_counter() - t0:.0f}s")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for seed, curve in dqn_curves.items():
    ax.plot(np.arange(1, len(curve) + 1) * 2500, curve, "o-", label=f"seed {seed}")
ax.axhline(500, color="gray", ls=":")
ax.set_xlabel("bước tương tác với môi trường")
ax.set_ylabel("return (chính sách tham lam)")
ax.set_title("DQN trên CartPole — 3 seed, cùng cấu hình")
ax.legend()
plt.tight_layout()
plt.show()

DQN **học được** từ con số 0 — vượt xa chính sách ngẫu nhiên (~24), có lúc giữ con lắc trọn 500 bước — nhưng **rất bất ổn**: cùng mã, cùng siêu tham số, chỉ khác seed, các đường cong khác hẳn nhau; một chính sách đã tốt có thể đột ngột sụp (khi bộ nhớ đầy các trạng thái "đang thăng bằng", mạng "quên" cách xử lý trạng thái nguy hiểm — catastrophic forgetting). Đây là thực tế của deep RL (Henderson et al., 2018 — *Deep Reinforcement Learning that Matters*):
- Luôn báo cáo **nhiều seed** (5–10) với khoảng tin cậy, không chọn seed đẹp nhất.
- **Đánh giá định kỳ và lưu checkpoint tốt nhất** (hàm `train_dqn` trả về `best_state`) thay vì lấy mô hình cuối.
- Siêu tham số (learning rate, tần suất cập nhật target, kích thước replay, lịch ε) ảnh hưởng rất lớn.

## 3. Policy gradient: tối ưu trực tiếp chính sách

Thay vì học giá trị rồi suy ra chính sách, tham số hoá **chính sách** $\pi_\theta(a\mid s)$ (mạng + softmax) và đi theo gradient của return kỳ vọng $J(\theta)$. **Định lý policy gradient** (log-derivative trick):

$$\nabla_\theta J(\theta) = \mathbb E_{\pi_\theta}\left[\sum_t \nabla_\theta\log\pi_\theta(a_t\mid s_t)\,G_t\right]$$

**REINFORCE** (Williams, 1992): chơi trọn một lượt, rồi tăng xác suất các hành động theo tỉ lệ return sau đó. Không cần max, xử lý tự nhiên hành động liên tục và chính sách ngẫu nhiên. Nhược điểm: ước lượng gradient có **phương sai rất lớn**. **Baseline** $b(s)$: dùng $G_t - b(s_t)$ thay cho $G_t$ — không đổi kỳ vọng gradient nhưng giảm phương sai; chọn $b(s) = V_\phi(s)$ học song song (advantage $A_t = G_t - V(s_t)$).

In [ ]:
def train_reinforce(use_baseline, episodes=600, gamma=0.99, seed=0):
    torch.manual_seed(seed)
    env = CartPole()
    env.reset(seed=seed)
    g = torch.Generator().manual_seed(seed)
    policy = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 2))
    value = nn.Sequential(nn.Linear(4, 64), nn.Tanh(), nn.Linear(64, 1))
    opt = torch.optim.Adam(list(policy.parameters()) + list(value.parameters()), lr=3e-3)
    lengths, steps = [], 0
    for ep in range(episodes):
        s, done, states, actions, rewards = env.reset(), False, [], [], []
        while not done:
            with torch.no_grad():
                a = torch.multinomial(F.softmax(policy(torch.tensor(s, dtype=torch.float32)), -1), 1, generator=g).item()
            s2, r, done = env.step(a)
            states.append(s)
            actions.append(a)
            rewards.append(r)
            s = s2
        steps += len(rewards)
        G, running = np.zeros(len(rewards)), 0.0
        for t in reversed(range(len(rewards))):                        # return từ mỗi bước
            running = rewards[t] + gamma * running
            G[t] = running
        S, G = torch.tensor(np.array(states), dtype=torch.float32), torch.tensor(G, dtype=torch.float32)
        log_probs = F.log_softmax(policy(S), -1)[torch.arange(len(actions)), torch.tensor(actions)]
        if use_baseline:
            v = value(S).squeeze(1)
            advantage = (G - v).detach()
            loss = -(log_probs * advantage).mean() + 0.005 * F.mse_loss(v, G)
        else:
            loss = -(log_probs * G).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
        lengths.append(len(rewards))
    return np.array(lengths), steps, policy


pg_results = {}
for use_baseline in [False, True]:
    t0 = time.perf_counter()
    lengths, steps, policy = train_reinforce(use_baseline)
    name = "REINFORCE + baseline" if use_baseline else "REINFORCE"
    with torch.no_grad():
        score = evaluate(lambda s: int(policy(torch.tensor(s, dtype=torch.float32)).argmax()))
    pg_results[name] = lengths
    print(f"{name:21s}: return TB 100 lượt cuối khi học = {lengths[-100:].mean():.0f} | đánh giá tham lam = {score:.0f} | "
          f"{steps:,} bước môi trường | {time.perf_counter() - t0:.0f}s")

fig, ax = plt.subplots(figsize=(8, 4))
for name, lengths in pg_results.items():
    ax.plot(np.convolve(lengths, np.ones(20) / 20, mode="valid"), label=name)
ax.set_xlabel("lượt")
ax.set_ylabel("return (TB trượt 20 lượt)")
ax.legend()
plt.tight_layout()
plt.show()

Baseline làm việc học nhanh và đều hơn rõ rệt: không có baseline, mọi hành động trong một lượt "khá tốt" đều được tăng xác suất như nhau (vì $G_t > 0$ luôn) — tín hiệu học rất nhiễu, đường cong dao động mạnh. Trừ $V(s)$ đi, chỉ hành động **tốt hơn mức kỳ vọng** ở trạng thái đó mới được khuyến khích.

So sánh hai họ:

| | Value-based (DQN) | Policy gradient (REINFORCE) |
|---|---|---|
| Học gì | $Q(s, a)$, chính sách = argmax | trực tiếp $\pi_\theta(a\mid s)$ |
| Dữ liệu | off-policy, replay → hiệu quả mẫu hơn | on-policy, mỗi lượt dùng một lần |
| Hành động liên tục | khó (cần max) | tự nhiên |
| Ổn định | bất ổn (deadly triad) | phương sai cao, cần baseline |

## 4. Các thuật toán hiện đại

- **Actor-critic** (A2C/A3C): actor = chính sách, critic = $V_\phi$; cập nhật mỗi bước bằng TD advantage $r + \gamma V(s') - V(s)$ thay vì chờ hết lượt.
- **PPO** (Schulman et al., 2017): giới hạn mỗi lần cập nhật chính sách không đổi quá nhiều (clipped objective) → ổn định, dễ dùng; là thuật toán mặc định của nhiều ứng dụng.
- **SAC, TD3**: actor-critic off-policy cho hành động liên tục (robot).
- **RLHF**: tinh chỉnh LLM — mô hình phần thưởng học từ so sánh của con người, rồi tối ưu chính sách (LLM) bằng PPO (hoặc các biến thể trực tiếp như DPO).
- **Model-based RL** (AlphaZero, MuZero, Dreamer): học/dùng mô hình môi trường để lập kế hoạch.

Thực tế: dùng thư viện đã kiểm chứng (Stable-Baselines3, CleanRL) với môi trường Gymnasium thay vì tự viết — tự viết để **hiểu**, như bài này.

## 🧠 Kiểm tra nhanh

1. Vì sao Q-learning với mạng nơ-ron ngây thơ hay phân kỳ? Experience replay và target network giúp gì?
2. Vì sao phải phân biệt "kết thúc vì thất bại" và "kết thúc vì hết giờ"?
3. Phát biểu định lý policy gradient. Vì sao REINFORCE có phương sai lớn?
4. Baseline giảm phương sai mà không làm lệch gradient — vì sao?
5. Vì sao phải báo cáo kết quả deep RL trên nhiều seed?

## 📌 Tóm tắt

- DQN = Q-learning + mạng nơ-ron + replay + target network (+ Double DQN); học được nhưng bất ổn — đánh giá nhiều seed, lưu checkpoint tốt nhất.
- Policy gradient tối ưu trực tiếp chính sách; REINFORCE + baseline (advantage) giảm phương sai; actor-critic/PPO là bước tiếp theo.
- RL hiện đại: PPO, SAC, model-based, RLHF cho LLM.

## 📚 Tài liệu tham khảo

- Mnih et al. (2015) — *Human-level control through deep reinforcement learning* (DQN)
- van Hasselt, Guez & Silver (2016) — *Deep Reinforcement Learning with Double Q-learning*
- Williams (1992) — *Simple statistical gradient-following algorithms for connectionist reinforcement learning* (REINFORCE)
- Schulman et al. (2017) — *Proximal Policy Optimization Algorithms*
- Henderson et al. (2018) — *Deep Reinforcement Learning that Matters*
- Ouyang et al. (2022) — *Training language models to follow instructions with human feedback* (RLHF)
- [OpenAI Spinning Up](https://spinningup.openai.com/), [CleanRL](https://github.com/vwxyzjn/cleanrl)